# Unified GTFS–NeTEx Calendar Comparison

I created this notebook to bring my calendar matching method together across all countries.

Two different things changed as I moved from country to country: **how I extracted the calendar data**, and **how I matched it**.

The extraction had to differ because the raw files are not organized the same way everywhere. For example:

- **France** defines calendar validity purely through `calendar_dates.txt`, with no `calendar.txt` weekday pattern at all, while the other countries use both together
- **Luxembourg** NeTEx calendar data comes as 700 small XML files
- **Sweden** comes as one single 682MB file
- **Italy** comes as one 590MB file with several tags packed onto a single line, which I had to reformat before I could even parse it

The DATA4PT GTFS-NeTEx mapping document confirms this kind of variation is expected. It describes **three different ways** a GTFS calendar record can be represented in NeTEx, depending on the implementation:

1. `DayType` alone
2. `PropertyOfDay` + `OperatingPeriod`
3. `DayType` + `DayTypeAssignment` + `OperatingPeriod`

So the differences I ran into reflect real, documented differences in how transport authorities choose to publish their NeTEx data.

The matching rule evolved over the course of the project. Some notebooks compared summary fields like active-day count and validity window, while others compared the real day-by-day bit pattern directly. The bit-pattern check is the stricter, more accurate method, and it is also what most countries already used (Norway, Sweden, and Italy used it from the start, and Luxembourg/France effectively landed on the same result). This notebook applies it consistently to every country, including Austria, which had not used it before.

**Matching rule:** each service's calendar is turned into a string of 1s and 0s, one digit per day in the shared comparison window

`1` = the service runs that day

`0` = service does not run that day

 A GTFS pattern and a NeTEx pattern count as a match only if that string is identical, day for day.

For example, say the window is five days, Monday to Friday:

**Bus A (GTFS)** runs Monday, Wednesday, and Friday -> so its pattern is `10101`

**Bus B (NeTEx)** also runs Monday, Wednesday, and Friday -> same pattern as A `10101` 

Since they share the same pattern they are counted as a match.

But now let us assume that Bus B instead ran every day that week, its pattern would be `11111`. Even though it still covers all of Bus A's days, the patterns are not identical anymore, so it would **not** count as a match.

**How each country's section works:** every section below first tries to load that country's `{gtfs_dates_by_id, netex_dates_by_id}` from `cache/<country>_dates.pkl`. If that cache file exists, it loads instantly. If it does not exist (for example, on a fresh clone of this repo before the cache has ever been built), the cell instead re-extracts everything from the raw GTFS/NeTEx files, using the same extraction logic described in that country's own original notebook, and saves the result to the cache for next time.

All third-party and standard-library imports used in this notebook are consolidated here.

In [1]:
import pickle
import zipfile
import gzip
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
from calendar_comparison import compare_calendar_patterns, summary_row

In [2]:
CACHE_DIR = Path("cache")
NS = {"n": "http://www.netex.org.uk/netex"}

def load_country(name):
    with open(CACHE_DIR / f"{name}_dates.pkl", "rb") as f:
        d = pickle.load(f)
    return d["gtfs_dates_by_id"], d["netex_dates_by_id"]

def local_name(tag):
    return tag.split("}", 1)[-1] if "}" in tag else tag

def read_gtfs_csv(zip_path, filename):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(filename) as f:
            return pd.read_csv(f, dtype=str)

def build_gtfs_dates_by_id_weekday(calendar, calendar_dates):
    """Standard GTFS reconstruction: calendar.txt weekday pattern with calendar_dates.txt
    exceptions applied. Used by Austria, Luxembourg, Norway, Sweden."""
    calendar = calendar.copy()
    calendar_dates = calendar_dates.copy()
    calendar["start_date"] = pd.to_datetime(calendar["start_date"], format="%Y%m%d")
    calendar["end_date"] = pd.to_datetime(calendar["end_date"], format="%Y%m%d")
    calendar_dates["date"] = pd.to_datetime(calendar_dates["date"], format="%Y%m%d")
    calendar_dates["exception_type"] = calendar_dates["exception_type"].astype(int)

    weekday_cols = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
    for c in weekday_cols:
        calendar[c] = calendar[c].astype(int)

    added, removed = defaultdict(set), defaultdict(set)
    for row in calendar_dates.itertuples(index=False):
        if row.exception_type == 1:
            added[row.service_id].add(row.date)
        elif row.exception_type == 2:
            removed[row.service_id].add(row.date)

    dates_by_id = {}
    for row in calendar.itertuples(index=False):
        sid = row.service_id
        all_days = pd.date_range(row.start_date, row.end_date, freq="D")
        active_weekdays = [i for i, c in enumerate(weekday_cols) if getattr(row, c) == 1]
        active = set(d for d in all_days if d.weekday() in active_weekdays)
        active = (active - removed[sid]) | added[sid]
        dates_by_id[sid] = active
    return dates_by_id

def build_gtfs_dates_by_id_exceptions_only(calendar_dates):
    """For GTFS feeds that define validity purely through calendar_dates.txt
    (exception_type == 1), with no calendar.txt weekday pattern at all.
    Used by France and Italy."""
    calendar_dates = calendar_dates.copy()
    calendar_dates["date"] = pd.to_datetime(calendar_dates["date"], format="%Y%m%d")
    calendar_dates["exception_type"] = calendar_dates["exception_type"].astype(int)
    return (
        calendar_dates[calendar_dates["exception_type"] == 1]
        .groupby("service_id")["date"]
        .apply(set)
        .to_dict()
    )

def extract_all_daytype_assignments(zip_path, file_list):
    """DayTypeAssignment -> {daytype_ref, operating_period_ref, date, is_available}.
    Used for Norway and Sweden, whose NeTEx calendars link DayTypeAssignment to a plain
    OperatingPeriod (date-range based, no ValidDayBits) rather than a UicOperatingPeriod.

    `is_available` comes from the <isAvailable> child on direct-date assignments: "false"
    means this is an explicit cancellation for that date (the NeTEx equivalent of GTFS
    calendar_dates.txt exception_type 2), while missing or "true" means an explicit
    addition (exception_type 1)."""
    rows = []
    with zipfile.ZipFile(zip_path) as z:
        for i, member in enumerate(file_list, 1):
            if i % 10 == 0 or i == len(file_list):
                print(f"  DayTypeAssignment: {i}/{len(file_list)}", end="\r")
            current, depth = None, 0
            with z.open(member) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)
                    if event == "start" and tag == "DayTypeAssignment":
                        depth += 1
                        if depth == 1:
                            current = {"daytype_ref": None, "operating_period_ref": None, "date": None, "is_available": None}
                    elif event == "end" and tag == "DayTypeAssignment":
                        if depth == 1 and current is not None:
                            rows.append(current)
                            current = None
                        depth -= 1
                        elem.clear()
                    elif event == "end" and current is not None and depth > 0:
                        if tag == "DayTypeRef" and current["daytype_ref"] is None:
                            current["daytype_ref"] = elem.attrib.get("ref")
                        elif tag == "OperatingPeriodRef" and current["operating_period_ref"] is None:
                            current["operating_period_ref"] = elem.attrib.get("ref")
                        elif tag == "Date" and current["date"] is None:
                            current["date"] = elem.text.strip() if elem.text else None
                        elif tag == "isAvailable" and current["is_available"] is None:
                            current["is_available"] = elem.text.strip() if elem.text else None
                        elem.clear()
                    elif event == "end":
                        elem.clear()
    print(f"\nDone -- {len(rows)} DayTypeAssignments extracted.")
    return pd.DataFrame(rows)

DAYSOFWEEK_MAP = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
                   "Friday": 4, "Saturday": 5, "Sunday": 6}

def expand_days_of_week(tokens):
    """Turn a DaysOfWeek token set (e.g. {\'Weekdays\'} or {\'Monday\', \'Friday\'}) into a
    set of Python weekday() integers (Monday=0 ... Sunday=6)."""
    out = set()
    for t in tokens:
        if t == "Everyday":
            out |= set(range(7))
        elif t == "Weekdays":
            out |= {0, 1, 2, 3, 4}
        elif t == "Weekend":
            out |= {5, 6}
        elif t in DAYSOFWEEK_MAP:
            out.add(DAYSOFWEEK_MAP[t])
    return out

def extract_all_daytype_restrictions(zip_path, file_list):
    """DayType -> DaysOfWeek restriction (a set of tokens, or None if the DayType has no
    weekly restriction at all). Read from properties/PropertyOfDay/DaysOfWeek.

    A plain OperatingPeriod only gives a date range with no day-by-day pattern; DaysOfWeek
    is what narrows that range down to specific weekdays, playing the same role GTFS's
    calendar.txt weekday columns play. Used for Norway and Sweden."""
    daytype_days = {}
    with zipfile.ZipFile(zip_path) as z:
        for i, member in enumerate(file_list, 1):
            if i % 10 == 0 or i == len(file_list):
                print(f"  DayType restrictions: {i}/{len(file_list)}", end="\r")
            current_id, depth = None, 0
            with z.open(member) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)
                    if event == "start" and tag == "DayType":
                        depth += 1
                        if depth == 1:
                            current_id = elem.attrib.get("id")
                            daytype_days.setdefault(current_id, None)
                    elif event == "end" and tag == "DayType":
                        depth -= 1
                        if depth == 0:
                            current_id = None
                        elem.clear()
                    elif event == "end" and tag == "DaysOfWeek" and current_id is not None:
                        text = elem.text.strip() if elem.text else None
                        if text:
                            tokens = set(text.split())
                            existing = daytype_days.get(current_id)
                            daytype_days[current_id] = tokens if existing is None else (existing | tokens)
                        elem.clear()
                    elif event == "end":
                        elem.clear()
    n_restricted = sum(1 for v in daytype_days.values() if v is not None)
    print(f"\nDone -- {len(daytype_days)} DayTypes seen, {n_restricted} with a DaysOfWeek restriction.")
    return daytype_days

def extract_all_operating_periods(zip_path, file_list):
    """OperatingPeriod -> {operating_period_id, from_date, to_date}. Used for Norway and Sweden."""
    rows = []
    with zipfile.ZipFile(zip_path) as z:
        for i, member in enumerate(file_list, 1):
            if i % 10 == 0 or i == len(file_list):
                print(f"  OperatingPeriod: {i}/{len(file_list)}", end="\r")
            current, depth = None, 0
            with z.open(member) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)
                    if event == "start" and tag == "OperatingPeriod":
                        depth += 1
                        if depth == 1:
                            current = {"operating_period_id": elem.attrib.get("id"), "from_date": None, "to_date": None}
                    elif event == "end" and tag == "OperatingPeriod":
                        if depth == 1 and current is not None:
                            rows.append(current)
                            current = None
                        depth -= 1
                        elem.clear()
                    elif event == "end" and current is not None and depth > 0:
                        text = elem.text.strip() if elem.text else None
                        if tag == "FromDate" and current["from_date"] is None:
                            current["from_date"] = text
                        elif tag == "ToDate" and current["to_date"] is None:
                            current["to_date"] = text
                        elem.clear()
                    elif event == "end":
                        elem.clear()
    print(f"\nDone -- {len(rows)} OperatingPeriods extracted.")
    return pd.DataFrame(rows)

results = {}

## Austria

**Extraction:** GTFS side from `calendar.txt` + `calendar_dates.txt` (regular weekday pattern with exceptions applied), producing one active-date set per `service_id`. NeTEx side from the ÖBB NeTEx feed's `ServiceJourney → DayTypeRef → DayTypeAssignment → UicOperatingPeriod` chain, keyed by `daytype_ref`, using the real `ValidDayBits` string on each `UicOperatingPeriod` (this raw bit data was already present in the original notebook's extraction, but the original notebook's final comparison only ever used a 5-field summary signature: validity start/end, active-day count, first/last active date and never actually compared the bits day-by-day).

**Raw data expected at:** `../DACH/MVD Austria/data/GTFS_Fahrplan_2026.zip` and `../DACH/MVD Austria/data/netex_evu_2026.zip` (see `DATA_SOURCES.md`).

In [3]:
cache_path = CACHE_DIR / "austria_dates.pkl"

if cache_path.exists():
    gtfs_dates_by_id, netex_dates_by_id = load_country("austria")
    print(f"Loaded from cache: {len(gtfs_dates_by_id)} GTFS ids, {len(netex_dates_by_id)} NeTEx ids")
else:
    print("Cache not found -- extracting Austria from raw data...")
    raw_dir = Path("../DACH/MVD Austria/data")
    gtfs_zip = raw_dir / "GTFS_Fahrplan_2026.zip"
    netex_zip = raw_dir / "netex_evu_2026.zip"

    calendar = read_gtfs_csv(gtfs_zip, "calendar.txt")
    calendar_dates = read_gtfs_csv(gtfs_zip, "calendar_dates.txt")
    gtfs_dates_by_id = build_gtfs_dates_by_id_weekday(calendar, calendar_dates)
    print(f"GTFS service ids: {len(gtfs_dates_by_id)}")

    # NeTEx side: ServiceJourney -> DayTypeRef -> DayTypeAssignment -> UicOperatingPeriod
    dta_rows, op_rows = [], []
    with zipfile.ZipFile(netex_zip) as z:
        xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]
        for i, name in enumerate(xml_files, 1):
            if i % 50 == 0 or i == len(xml_files):
                print(f"  {i}/{len(xml_files)} files...", end="\r")
            with z.open(name) as f:
                try:
                    root = ET.parse(f).getroot()
                except ET.ParseError:
                    continue
            for dta in root.findall(".//n:DayTypeAssignment", NS):
                dref = dta.find("n:DayTypeRef", NS)
                oref = dta.find("n:OperatingPeriodRef", NS)
                dta_rows.append({
                    "daytype_ref": dref.attrib.get("ref") if dref is not None else None,
                    "operating_period_ref": oref.attrib.get("ref") if oref is not None else None,
                })
            for op in root.findall(".//n:UicOperatingPeriod", NS):
                op_rows.append({
                    "operating_period_id": op.attrib.get("id"),
                    "from_date": op.findtext("n:FromDate", default=None, namespaces=NS),
                    "valid_day_bits": op.findtext("n:ValidDayBits", default=None, namespaces=NS),
                })
    print(f"\nDayTypeAssignment rows: {len(dta_rows)}, UicOperatingPeriod rows: {len(op_rows)}")

    dta_df = pd.DataFrame(dta_rows).dropna(subset=["daytype_ref"]).drop_duplicates(subset=["daytype_ref", "operating_period_ref"])
    op_df = pd.DataFrame(op_rows).drop_duplicates(subset=["operating_period_id"])
    joined = dta_df.merge(op_df, left_on="operating_period_ref", right_on="operating_period_id", how="left")
    joined["from_date"] = pd.to_datetime(joined["from_date"].str[:10], errors="coerce")

    netex_dates_by_id = {}
    for row in joined.itertuples(index=False):
        if pd.isna(row.from_date) or not row.valid_day_bits or isinstance(row.valid_day_bits, float):
            continue
        active = {row.from_date + pd.Timedelta(days=i) for i, b in enumerate(row.valid_day_bits) if b == "1"}
        if active:
            netex_dates_by_id[row.daytype_ref] = active
    print(f"NeTEx daytype_ref ids: {len(netex_dates_by_id)}")

    CACHE_DIR.mkdir(exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump({"gtfs_dates_by_id": gtfs_dates_by_id, "netex_dates_by_id": netex_dates_by_id}, f)

results["Austria"] = compare_calendar_patterns(gtfs_dates_by_id, netex_dates_by_id)
r = results["Austria"]
print(f"GTFS ids: {r['gtfs_ids_total']:,}   NeTEx ids: {r['netex_ids_total']:,}")
print(f"Unique GTFS patterns: {r['gtfs_unique_patterns']:,}   Unique NeTEx patterns: {r['netex_unique_patterns']:,}")
print(f"Matched patterns: {r['matched_patterns']:,}")
print(f"GTFS match rate: {r['gtfs_match_rate_pct']}%   NeTEx match rate: {r['netex_match_rate_pct']}%")

Loaded from cache: 1340 GTFS ids, 4111 NeTEx ids
GTFS ids: 1,340   NeTEx ids: 4,111
Unique GTFS patterns: 1,338   Unique NeTEx patterns: 4,111
Matched patterns: 764
GTFS match rate: 57.1%   NeTEx match rate: 18.58%


## Luxembourg

**Extraction:** GTFS side from `calendar.txt` + `calendar_dates.txt`, one active-date set per `service_id`. NeTEx side parses all 700 XML files in the NeTEx feed, extracting `DayTypeAssignment → UicOperatingPeriod` pairs keyed by `assignment_id`, and builds active dates from each `UicOperatingPeriod`'s `FromDate` + `ValidDayBits`.

The original Luxembourg notebook used a 4-field match (active-day count, first/last active date, and `valid_day_bits`) rather than bits alone. 
**Raw data expected at:** `../Western Europe/data/gtfs-20260408-20260430.zip` and `../Western Europe/data/netex-20260408-20260430.zip`.

In [4]:
cache_path = CACHE_DIR / "luxembourg_dates.pkl"

if cache_path.exists():
    gtfs_dates_by_id, netex_dates_by_id = load_country("luxembourg")
    print(f"Loaded from cache: {len(gtfs_dates_by_id)} GTFS ids, {len(netex_dates_by_id)} NeTEx ids")
else:
    print("Cache not found -- extracting Luxembourg from raw data...")
    data_dir = Path("../Western Europe/data")
    gtfs_zip = data_dir / "gtfs-20260408-20260430.zip"
    netex_zip = data_dir / "netex-20260408-20260430.zip"

    calendar = read_gtfs_csv(gtfs_zip, "calendar.txt")
    calendar_dates = read_gtfs_csv(gtfs_zip, "calendar_dates.txt")
    gtfs_dates_by_id = build_gtfs_dates_by_id_weekday(calendar, calendar_dates)
    print(f"GTFS service ids: {len(gtfs_dates_by_id)}")

    dta_rows, op_rows = [], []
    with zipfile.ZipFile(netex_zip) as z:
        xml_files = sorted(n for n in z.namelist() if n.lower().endswith(".xml"))
        for i, name in enumerate(xml_files, 1):
            if i % 50 == 0 or i == len(xml_files):
                print(f"  {i}/{len(xml_files)} files...", end="\r")
            with z.open(name) as f:
                try:
                    root = ET.parse(f).getroot()
                except ET.ParseError:
                    continue
            for dta in root.findall(".//n:DayTypeAssignment", NS):
                dref = dta.find("n:DayTypeRef", NS)
                oref = dta.find("n:OperatingPeriodRef", NS)
                dta_rows.append({
                    "assignment_id": dta.attrib.get("id"),
                    "daytype_ref": dref.attrib.get("ref") if dref is not None else None,
                    "operating_period_ref": oref.attrib.get("ref") if oref is not None else None,
                })
            for op in root.findall(".//n:UicOperatingPeriod", NS):
                op_rows.append({
                    "operating_period_id": op.attrib.get("id"),
                    "from_date": op.findtext("n:FromDate", default=None, namespaces=NS),
                    "valid_day_bits": op.findtext("n:ValidDayBits", default=None, namespaces=NS),
                })
    print(f"\nDayTypeAssignment rows: {len(dta_rows)}, UicOperatingPeriod rows: {len(op_rows)}")

    dta_df = pd.DataFrame(dta_rows).drop_duplicates(subset=["assignment_id"])
    op_df = pd.DataFrame(op_rows).drop_duplicates(subset=["operating_period_id"])
    joined = dta_df.merge(op_df, left_on="operating_period_ref", right_on="operating_period_id", how="left")
    joined["from_date"] = pd.to_datetime(joined["from_date"].str[:10], errors="coerce")

    netex_dates_by_id = {}
    for row in joined.itertuples(index=False):
        if pd.isna(row.from_date) or not row.valid_day_bits or isinstance(row.valid_day_bits, float):
            continue
        active = {row.from_date + pd.Timedelta(days=i) for i, b in enumerate(row.valid_day_bits) if b == "1"}
        if active:
            netex_dates_by_id[row.assignment_id] = active
    print(f"NeTEx assignment ids: {len(netex_dates_by_id)}")

    CACHE_DIR.mkdir(exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump({"gtfs_dates_by_id": gtfs_dates_by_id, "netex_dates_by_id": netex_dates_by_id}, f)

results["Luxembourg"] = compare_calendar_patterns(gtfs_dates_by_id, netex_dates_by_id)
r = results["Luxembourg"]
print(f"GTFS ids: {r['gtfs_ids_total']:,}   NeTEx ids: {r['netex_ids_total']:,}")
print(f"Unique GTFS patterns: {r['gtfs_unique_patterns']:,}   Unique NeTEx patterns: {r['netex_unique_patterns']:,}")
print(f"Matched patterns: {r['matched_patterns']:,}")
print(f"GTFS match rate: {r['gtfs_match_rate_pct']}%   NeTEx match rate: {r['netex_match_rate_pct']}%")

Loaded from cache: 100 GTFS ids, 2042 NeTEx ids
GTFS ids: 100   NeTEx ids: 2,042
Unique GTFS patterns: 100   Unique NeTEx patterns: 115
Matched patterns: 81
GTFS match rate: 81.0%   NeTEx match rate: 70.43%


## France

**Extraction:** GTFS side from `calendar_dates.txt` only (France's GTFS feed defines service validity purely through explicit date exceptions, `exception_type == 1`), one active-date set per `service_id`. NeTEx side streams `UicOperatingPeriod` elements directly out of the largest XML member of the raw NeTEx zip (a ~545MB file), keyed by `operating_period_id`, using each element's `FromDate` + `ValidDayBits`.

France's own notebook already included a bits-only sensitivity check (99.99% / 98.04%) alongside its main 4-field-match result. The result below is expected to match that sensitivity check almost exactly; the module intentionally excludes all-zero (empty-window) patterns from the unique-pattern count, which can cause a difference of at most one pattern from the original notebook's count.

**Raw data expected at:** `../Western Europe/data/Export_OpenData_SNCF_GTFS_NewTripId.zip` and `../Western Europe/data/export-opendata-sncf-netex.zip`.

In [5]:
cache_path = CACHE_DIR / "france_dates.pkl"

if cache_path.exists():
    gtfs_dates_by_id, netex_dates_by_id = load_country("france")
    print(f"Loaded from cache: {len(gtfs_dates_by_id)} GTFS ids, {len(netex_dates_by_id)} NeTEx ids")
else:
    print("Cache not found -- extracting France from raw data (parses a ~545MB file, can take a few minutes)...")
    data_dir = Path("../Western Europe/data")
    gtfs_zip = data_dir / "Export_OpenData_SNCF_GTFS_NewTripId.zip"
    netex_zip = data_dir / "export-opendata-sncf-netex.zip"

    calendar_dates = read_gtfs_csv(gtfs_zip, "calendar_dates.txt")
    gtfs_dates_by_id = build_gtfs_dates_by_id_exceptions_only(calendar_dates)
    print(f"GTFS service ids: {len(gtfs_dates_by_id)}")

    op_rows = []
    current, depth = None, 0
    with zipfile.ZipFile(netex_zip) as z:
        xml_members = [n for n in z.namelist() if n.lower().endswith(".xml")]
        member = max(xml_members, key=lambda n: z.getinfo(n).file_size)
        print(f"Parsing {member} ...")
        with z.open(member) as f:
            for event, elem in ET.iterparse(f, events=("start", "end")):
                tag = local_name(elem.tag)
                if event == "start" and tag == "UicOperatingPeriod":
                    depth += 1
                    if depth == 1:
                        current = {"operating_period_id": elem.attrib.get("id"), "from_date": None, "valid_day_bits": None}
                elif event == "end" and tag == "UicOperatingPeriod":
                    if depth == 1 and current is not None:
                        op_rows.append(current)
                        current = None
                    depth -= 1
                    elem.clear()
                elif event == "end" and current is not None and depth > 0:
                    text = elem.text.strip() if elem.text else None
                    if tag == "FromDate" and current["from_date"] is None:
                        current["from_date"] = text
                    elif tag == "ValidDayBits" and current["valid_day_bits"] is None:
                        current["valid_day_bits"] = text
                    elem.clear()
                elif event == "end":
                    elem.clear()
    print(f"UicOperatingPeriod rows: {len(op_rows)}")

    op_df = pd.DataFrame(op_rows)
    op_df["from_date"] = pd.to_datetime(op_df["from_date"].str[:10], errors="coerce")

    netex_dates_by_id = {}
    for row in op_df.itertuples(index=False):
        if pd.isna(row.from_date) or not row.valid_day_bits:
            continue
        active = {row.from_date + pd.Timedelta(days=i) for i, b in enumerate(row.valid_day_bits) if b == "1"}
        if active:
            netex_dates_by_id[row.operating_period_id] = active
    print(f"NeTEx operating_period ids: {len(netex_dates_by_id)}")

    CACHE_DIR.mkdir(exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump({"gtfs_dates_by_id": gtfs_dates_by_id, "netex_dates_by_id": netex_dates_by_id}, f)

results["France"] = compare_calendar_patterns(gtfs_dates_by_id, netex_dates_by_id)
r = results["France"]
print(f"GTFS ids: {r['gtfs_ids_total']:,}   NeTEx ids: {r['netex_ids_total']:,}")
print(f"Unique GTFS patterns: {r['gtfs_unique_patterns']:,}   Unique NeTEx patterns: {r['netex_unique_patterns']:,}")
print(f"Matched patterns: {r['matched_patterns']:,}")
print(f"GTFS match rate: {r['gtfs_match_rate_pct']}%   NeTEx match rate: {r['netex_match_rate_pct']}%")

Loaded from cache: 12815 GTFS ids, 11427 NeTEx ids
GTFS ids: 12,815   NeTEx ids: 11,427
Unique GTFS patterns: 11,203   Unique NeTEx patterns: 11,427
Matched patterns: 11,203
GTFS match rate: 100.0%   NeTEx match rate: 98.04%


## Norway

**Extraction:** GTFS side from `calendar.txt` + `calendar_dates.txt`. NeTEx side parses `DayTypeAssignment`, `OperatingPeriod`, and `DayType` elements from the `_shared_data` NeTEx files (the ones that actually contain calendar data, out of ~5,000 files in the full feed), keyed by `daytype_ref`.

**Correction applied here, not present in the original notebook:** a plain NeTEx `OperatingPeriod` only gives a date range (`from_date`/`to_date`), not a day-by-day pattern. The original notebook treated every day inside that range as active. Checking the raw data directly showed two things were missing:

1. `DayType` elements can carry a `DaysOfWeek` restriction (e.g. "only Monday-Friday") that narrows which days inside the range are actually active -> the NeTEx equivalent of GTFS's `calendar.txt` weekday columns.
2. Individual `DayTypeAssignment` entries with a direct date and `isAvailable=false` are explicit cancellations (the NeTEx equivalent of `calendar_dates.txt` `exception_type=2`) -> the original notebook counted these as active days instead of excluding them.

Both are applied below: period-based dates are filtered to each `DayType`'s allowed weekdays, and direct-date assignments are split into real additions (`isAvailable != false`) and real cancellations (`isAvailable == false`), which are then subtracted. 

**Raw data expected at:** `../Northern Europe/data/rb_norway-aggregated-gtfs.zip` and `../Northern Europe/data/rb_norway-aggregated-netex.zip`.

In [6]:
cache_path = CACHE_DIR / "norway_dates.pkl"

if cache_path.exists():
    gtfs_dates_by_id, netex_dates_by_id = load_country("norway")
    print(f"Loaded from cache: {len(gtfs_dates_by_id)} GTFS ids, {len(netex_dates_by_id)} NeTEx ids")
else:
    print("Cache not found -- extracting Norway from raw data...")
    data_dir = Path("../Northern Europe/data")
    gtfs_zip = data_dir / "rb_norway-aggregated-gtfs.zip"
    netex_zip = data_dir / "rb_norway-aggregated-netex.zip"

    calendar = read_gtfs_csv(gtfs_zip, "calendar.txt")
    calendar_dates = read_gtfs_csv(gtfs_zip, "calendar_dates.txt")
    gtfs_dates_by_id = build_gtfs_dates_by_id_weekday(calendar, calendar_dates)
    print(f"GTFS service ids: {len(gtfs_dates_by_id)}")

    with zipfile.ZipFile(netex_zip) as z:
        netex_file_names = [n for n in z.namelist() if n.lower().endswith(".xml")]
    shared_files = [f for f in netex_file_names if "shared_data" in f]
    print(f"Shared data files: {len(shared_files)}")

    netex_daytype_assignments = extract_all_daytype_assignments(netex_zip, shared_files)
    netex_daytype_assignments["date"] = pd.to_datetime(netex_daytype_assignments["date"], errors="coerce")

    netex_operating_periods = extract_all_operating_periods(netex_zip, shared_files)
    netex_operating_periods["from_date"] = pd.to_datetime(netex_operating_periods["from_date"], errors="coerce")
    netex_operating_periods["to_date"] = pd.to_datetime(netex_operating_periods["to_date"], errors="coerce")

    daytype_days = extract_all_daytype_restrictions(netex_zip, shared_files)

    # 1. Direct-date assignments: split into real additions (isAvailable != "false") and
    # explicit cancellations (isAvailable == "false") -- the NeTEx equivalent of GTFS's
    # calendar_dates.txt exception_type 1 (added) vs 2 (removed). The original notebook
    # treated every direct-date assignment as an addition, regardless of isAvailable.
    direct = netex_daytype_assignments[netex_daytype_assignments["date"].notna()]
    netex_added_dates = (
        direct[direct["is_available"] != "false"]
        [["daytype_ref", "date"]].rename(columns={"date": "active_date"})
    )
    netex_removed_dates = (
        direct[direct["is_available"] == "false"]
        [["daytype_ref", "date"]].rename(columns={"date": "active_date"})
    )

    # 2. Operating-period-based assignments, expanded and narrowed to each DayType's
    # DaysOfWeek restriction (vectorized -- the original notebook expanded these one row
    # at a time with a Python loop, which is too slow at this scale, and did not apply any
    # weekday restriction at all).
    period_assignments = netex_daytype_assignments[
        netex_daytype_assignments["date"].isna() & netex_daytype_assignments["operating_period_ref"].notna()
    ].merge(
        netex_operating_periods[["operating_period_id", "from_date", "to_date"]],
        left_on="operating_period_ref", right_on="operating_period_id", how="left"
    )
    period_assignments = period_assignments[
        period_assignments["from_date"].notna() & period_assignments["to_date"].notna()
    ].copy()

    def expand_period_row(row):
        full_range = pd.date_range(row["from_date"], row["to_date"], freq="D")
        restriction = daytype_days.get(row["daytype_ref"])
        if restriction is None:
            return full_range
        allowed = expand_days_of_week(restriction)
        return full_range[full_range.weekday.isin(list(allowed))]

    period_assignments["date_range"] = period_assignments.apply(expand_period_row, axis=1)
    netex_period_dates = (
        period_assignments[["daytype_ref", "date_range"]]
        .explode("date_range")
        .rename(columns={"date_range": "active_date"})
    )

    # 3. Combine: weekday-filtered period dates plus real additions, minus real cancellations
    netex_base_dates = pd.concat(
        [netex_period_dates, netex_added_dates], ignore_index=True
    ).drop_duplicates(subset=["daytype_ref", "active_date"])
    netex_base_dates["active_date"] = pd.to_datetime(netex_base_dates["active_date"], errors="coerce").dt.normalize()
    netex_removed_dates = netex_removed_dates.copy()
    netex_removed_dates["active_date"] = pd.to_datetime(netex_removed_dates["active_date"], errors="coerce").dt.normalize()

    base_by_id = netex_base_dates.dropna(subset=["daytype_ref", "active_date"]).groupby("daytype_ref")["active_date"].apply(set).to_dict()
    removed_by_id = netex_removed_dates.dropna(subset=["daytype_ref", "active_date"]).groupby("daytype_ref")["active_date"].apply(set).to_dict()

    netex_dates_by_id = {}
    for did, dates in base_by_id.items():
        final_dates = dates - removed_by_id.get(did, set())
        if final_dates:
            netex_dates_by_id[did] = final_dates
    print(f"NeTEx daytype_ref ids: {len(netex_dates_by_id)}")

    CACHE_DIR.mkdir(exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump({"gtfs_dates_by_id": gtfs_dates_by_id, "netex_dates_by_id": netex_dates_by_id}, f)

results["Norway"] = compare_calendar_patterns(gtfs_dates_by_id, netex_dates_by_id)
r = results["Norway"]
print(f"GTFS ids: {r['gtfs_ids_total']:,}   NeTEx ids: {r['netex_ids_total']:,}")
print(f"Unique GTFS patterns: {r['gtfs_unique_patterns']:,}   Unique NeTEx patterns: {r['netex_unique_patterns']:,}")
print(f"Matched patterns: {r['matched_patterns']:,}")
print(f"GTFS match rate: {r['gtfs_match_rate_pct']}%   NeTEx match rate: {r['netex_match_rate_pct']}%")

Loaded from cache: 9854 GTFS ids, 15524 NeTEx ids
GTFS ids: 9,854   NeTEx ids: 15,524
Unique GTFS patterns: 4,791   Unique NeTEx patterns: 3,057
Matched patterns: 2,789
GTFS match rate: 58.21%   NeTEx match rate: 91.23%


## Sweden

**Extraction:** GTFS side from `calendar.txt` + `calendar_dates.txt`. NeTEx side reuses the exact same `DayTypeAssignment`/`OperatingPeriod`/`DayType` extraction as Norway (Sweden's notebook explicitly reused Norway's extraction code), applied to Sweden's single 715MB `_shared_data.xml` file, keyed by `daytype_ref`.

**Same correction as Norway, and it matters even more here:** Sweden's calendar data is dominated by direct-date `DayTypeAssignment` entries rather than period-based ones, and most of them (about 76%) are `isAvailable=false` cancellations that the original notebook counted as active days. The same `DaysOfWeek` and `isAvailable` correction described in the Norway section above is applied here. 

**Raw data expected at:** `../Northern Europe/data/GTFS Sverige 2.zip` and `../Northern Europe/data/NeTEx Sweden Static data.zip`.

In [7]:
cache_path = CACHE_DIR / "sweden_dates.pkl"

if cache_path.exists():
    gtfs_dates_by_id, netex_dates_by_id = load_country("sweden")
    print(f"Loaded from cache: {len(gtfs_dates_by_id)} GTFS ids, {len(netex_dates_by_id)} NeTEx ids")
else:
    print("Cache not found -- extracting Sweden from raw data (parses a 682MB file, can take several minutes)...")
    data_dir = Path("../Northern Europe/data")
    gtfs_zip = data_dir / "GTFS Sverige 2.zip"
    netex_zip = data_dir / "NeTEx Sweden Static data.zip"

    calendar = read_gtfs_csv(gtfs_zip, "calendar.txt")
    calendar_dates = read_gtfs_csv(gtfs_zip, "calendar_dates.txt")
    gtfs_dates_by_id = build_gtfs_dates_by_id_weekday(calendar, calendar_dates)
    print(f"GTFS service ids: {len(gtfs_dates_by_id)}")

    with zipfile.ZipFile(netex_zip) as z:
        shared_files = [n for n in z.namelist() if n.lower().endswith(".xml")]

    netex_daytype_assignments = extract_all_daytype_assignments(netex_zip, shared_files)
    netex_daytype_assignments["date"] = pd.to_datetime(netex_daytype_assignments["date"], errors="coerce")

    netex_operating_periods = extract_all_operating_periods(netex_zip, shared_files)
    netex_operating_periods["from_date"] = pd.to_datetime(netex_operating_periods["from_date"], errors="coerce")
    netex_operating_periods["to_date"] = pd.to_datetime(netex_operating_periods["to_date"], errors="coerce")

    daytype_days = extract_all_daytype_restrictions(netex_zip, shared_files)

    # 1. Direct-date assignments: split into real additions (isAvailable != "false") and
    # explicit cancellations (isAvailable == "false"). For Sweden this is the dominant
    # source of calendar data -- most DayTypeAssignments here are direct-date, not
    # period-based, and most of those are cancellations the original notebook miscounted.
    direct = netex_daytype_assignments[netex_daytype_assignments["date"].notna()]
    netex_added_dates = (
        direct[direct["is_available"] != "false"]
        [["daytype_ref", "date"]].rename(columns={"date": "active_date"})
    )
    netex_removed_dates = (
        direct[direct["is_available"] == "false"]
        [["daytype_ref", "date"]].rename(columns={"date": "active_date"})
    )

    # 2. Operating-period-based assignments, expanded and narrowed to each DayType's
    # DaysOfWeek restriction (vectorized -- see the Norway cell above for the same pattern).
    period_assignments = netex_daytype_assignments[
        netex_daytype_assignments["date"].isna() & netex_daytype_assignments["operating_period_ref"].notna()
    ].merge(
        netex_operating_periods[["operating_period_id", "from_date", "to_date"]],
        left_on="operating_period_ref", right_on="operating_period_id", how="left"
    )
    period_assignments = period_assignments[
        period_assignments["from_date"].notna() & period_assignments["to_date"].notna()
    ].copy()

    def expand_period_row(row):
        full_range = pd.date_range(row["from_date"], row["to_date"], freq="D")
        restriction = daytype_days.get(row["daytype_ref"])
        if restriction is None:
            return full_range
        allowed = expand_days_of_week(restriction)
        return full_range[full_range.weekday.isin(list(allowed))]

    period_assignments["date_range"] = period_assignments.apply(expand_period_row, axis=1)
    netex_period_dates = (
        period_assignments[["daytype_ref", "date_range"]]
        .explode("date_range")
        .rename(columns={"date_range": "active_date"})
    )

    # 3. Combine: weekday-filtered period dates plus real additions, minus real cancellations
    netex_base_dates = pd.concat(
        [netex_period_dates, netex_added_dates], ignore_index=True
    ).drop_duplicates(subset=["daytype_ref", "active_date"])
    netex_base_dates["active_date"] = pd.to_datetime(netex_base_dates["active_date"], errors="coerce").dt.normalize()
    netex_removed_dates = netex_removed_dates.copy()
    netex_removed_dates["active_date"] = pd.to_datetime(netex_removed_dates["active_date"], errors="coerce").dt.normalize()

    base_by_id = netex_base_dates.dropna(subset=["daytype_ref", "active_date"]).groupby("daytype_ref")["active_date"].apply(set).to_dict()
    removed_by_id = netex_removed_dates.dropna(subset=["daytype_ref", "active_date"]).groupby("daytype_ref")["active_date"].apply(set).to_dict()

    netex_dates_by_id = {}
    for did, dates in base_by_id.items():
        final_dates = dates - removed_by_id.get(did, set())
        if final_dates:
            netex_dates_by_id[did] = final_dates
    print(f"NeTEx daytype_ref ids: {len(netex_dates_by_id)}")

    CACHE_DIR.mkdir(exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump({"gtfs_dates_by_id": gtfs_dates_by_id, "netex_dates_by_id": netex_dates_by_id}, f)

results["Sweden"] = compare_calendar_patterns(gtfs_dates_by_id, netex_dates_by_id)
r = results["Sweden"]
print(f"GTFS ids: {r['gtfs_ids_total']:,}   NeTEx ids: {r['netex_ids_total']:,}")
print(f"Unique GTFS patterns: {r['gtfs_unique_patterns']:,}   Unique NeTEx patterns: {r['netex_unique_patterns']:,}")
print(f"Matched patterns: {r['matched_patterns']:,}")
print(f"GTFS match rate: {r['gtfs_match_rate_pct']}%   NeTEx match rate: {r['netex_match_rate_pct']}%")

Loaded from cache: 6892 GTFS ids, 7516 NeTEx ids
GTFS ids: 6,892   NeTEx ids: 7,516
Unique GTFS patterns: 6,892   Unique NeTEx patterns: 7,398
Matched patterns: 4,027
GTFS match rate: 58.43%   NeTEx match rate: 54.43%


## Italy

**Extraction:** GTFS side from `calendar_dates.txt` only, same as France. NeTEx side streams `UicOperatingPeriod` elements directly out of Italy's 590MB gzipped NeTEx file. Italy's NeTEx feed does not use a separate `DayTypeAssignment` join at all, each `UicOperatingPeriod`'s id matches its corresponding `DayType` id directly (e.g. `...UicOperatingPeriod:TRENITALIA:10-5A01-0083-6` corresponds to `...DayType:TRENITALIA:10-5A01-0083-6`), so the `DayType` id is derived by a simple string replacement rather than a table join.

**Raw data expected at:** `../Italy and Spain/data/trenitalia.gtfs` and `../Italy and Spain/data/IT-IT-TRENITALIA_L1.xml.gz`.

In [8]:
cache_path = CACHE_DIR / "italy_dates.pkl"

if cache_path.exists():
    gtfs_dates_by_id, netex_dates_by_id = load_country("italy")
    print(f"Loaded from cache: {len(gtfs_dates_by_id)} GTFS ids, {len(netex_dates_by_id)} NeTEx ids")
else:
    print("Cache not found -- extracting Italy from raw data (parses a 590MB gzip file, can take a few minutes)...")
    data_dir = Path("../Italy and Spain/data")
    gtfs_zip = data_dir / "trenitalia.gtfs"
    netex_path = data_dir / "IT-IT-TRENITALIA_L1.xml.gz"

    calendar_dates = read_gtfs_csv(gtfs_zip, "calendar_dates.txt")
    gtfs_dates_by_id = build_gtfs_dates_by_id_exceptions_only(calendar_dates)
    print(f"GTFS service ids: {len(gtfs_dates_by_id)}")

    op_rows = []
    current, depth = None, 0
    with gzip.open(netex_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("start", "end")):
            tag = local_name(elem.tag)
            if event == "start" and tag == "UicOperatingPeriod":
                depth += 1
                if depth == 1:
                    current = {"uic_period_id": elem.attrib.get("id"), "from_date": None, "valid_day_bits": None}
            elif event == "end" and tag == "UicOperatingPeriod":
                if depth == 1 and current is not None:
                    op_rows.append(current)
                    current = None
                depth -= 1
                elem.clear()
            elif event == "end" and current is not None and depth > 0:
                text = elem.text.strip() if elem.text else None
                if tag == "FromDate" and current["from_date"] is None:
                    current["from_date"] = text
                elif tag == "ValidDayBits" and current["valid_day_bits"] is None:
                    current["valid_day_bits"] = text
                elem.clear()
            elif event == "end":
                elem.clear()
    print(f"UicOperatingPeriod rows: {len(op_rows)}")

    netex_uic_periods = pd.DataFrame(op_rows)
    netex_uic_periods["from_date"] = pd.to_datetime(netex_uic_periods["from_date"], errors="coerce").dt.normalize()

    # Bit -> date expansion, vectorized with numpy (the original notebook did this with a slow
    # per-row, per-character Python loop; this does the same thing, just faster at 33K+ rows)
    netex_dates_by_id = {}
    for row in netex_uic_periods.itertuples(index=False):
        bits = row.valid_day_bits
        from_date = row.from_date
        if not bits or pd.isna(from_date):
            continue
        # Derive the matching DayType id by replacing UicOperatingPeriod with DayType in the id string
        daytype_ref = row.uic_period_id.replace("UicOperatingPeriod", "DayType")
        offsets = np.frombuffer(bits.encode(), dtype=np.uint8) - ord("0")
        active_offsets = np.nonzero(offsets == 1)[0]
        if len(active_offsets):
            netex_dates_by_id[daytype_ref] = {from_date + pd.Timedelta(days=int(o)) for o in active_offsets}
    print(f"NeTEx daytype_ref ids: {len(netex_dates_by_id)}")

    CACHE_DIR.mkdir(exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump({"gtfs_dates_by_id": gtfs_dates_by_id, "netex_dates_by_id": netex_dates_by_id}, f)

results["Italy"] = compare_calendar_patterns(gtfs_dates_by_id, netex_dates_by_id)
r = results["Italy"]
print(f"GTFS ids: {r['gtfs_ids_total']:,}   NeTEx ids: {r['netex_ids_total']:,}")
print(f"Unique GTFS patterns: {r['gtfs_unique_patterns']:,}   Unique NeTEx patterns: {r['netex_unique_patterns']:,}")
print(f"Matched patterns: {r['matched_patterns']:,}")
print(f"GTFS match rate: {r['gtfs_match_rate_pct']}%   NeTEx match rate: {r['netex_match_rate_pct']}%")

Loaded from cache: 231 GTFS ids, 33054 NeTEx ids
GTFS ids: 231   NeTEx ids: 33,054
Unique GTFS patterns: 87   Unique NeTEx patterns: 5,431
Matched patterns: 44
GTFS match rate: 50.57%   NeTEx match rate: 0.81%


## Combined summary

In [9]:
summary_df = pd.DataFrame([summary_row(name, r) for name, r in results.items()])
summary_df

,country,shared_window_days,gtfs_unique_patterns,netex_unique_patterns,matched_patterns,gtfs_match_rate_pct,netex_match_rate_pct
0,Austria,364,1338,4111,764,57.10,18.58
1,Luxembourg,23,100,115,81,81.00,70.43
2,France,151,11203,11427,11203,100.00,98.04
3,Norway,2989,4791,3057,2789,58.21,91.23
4,Sweden,246,6892,7398,4027,58.43,54.43
5,Italy,150,87,5431,44,50.57,0.81
